<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/1working_progress2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas
!pip install xlsxwriter

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import os
import zipfile

In [ ]:
def temizle_ve_cevir(x):
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

In [ ]:
# Temizlenecek değerler listesi
temizlenecekler = ["nan", "Nan", "NaN", ""," ","0","naN","nAn","NAn","nAN"]

def temizle_nan(value):
    if str(value).strip().lower() in [x.lower() for x in temizlenecekler]:
        return pd.NA
    return value


In [ ]:
# Güvenli filtreleme fonksiyonu
def dolu_mi(x):
    return pd.notna(x) and str(x).strip() != "" and str(x).strip().upper() != ""


In [ ]:
B3_status = pd.read_excel('/content/B3_status-2026-2022 RU.xlsx')

In [ ]:
me2n = pd.read_excel('/content/me2n_2026_2022.xlsx')

In [ ]:
me5a = pd.read_excel('/content/me5a_2026_2022.xlsx')

In [40]:
import pandas as pd

try:
    # TCMB günlük kur XML adresi
    url = "https://www.tcmb.gov.tr/kurlar/today.xml"
    df = pd.read_xml(url, xpath="//Currency")

    capraz_kur_listesi = []

    for _, row in df.iterrows():
        code = row.get('CurrencyCode')
        isim = row.get('Isim')
        cross_usd = row.get('CrossRateUSD')
        cross_other = row.get('CrossRateOther')

        if code and code != 'TRY':
            # 1. XML'de doğrudan tanımlı USD çapraz kurları varsa ekle
            if pd.notna(cross_usd) and str(cross_usd).strip() != "":
                capraz_kur_listesi.append({
                    'Para Birimi': code,
                    'Tanım': isim,
                    'Çapraz Kur Tipi': f"{code}/USD",
                    'Değer': float(cross_usd),
                    'Kaynak': 'TCMB Doğrudan Veri'
                })

            if pd.notna(cross_other) and str(cross_other).strip() != "":
                capraz_kur_listesi.append({
                    'Para Birimi': code,
                    'Tanım': isim,
                    'Çapraz Kur Tipi': f"USD/{code}",
                    'Değer': float(cross_other),
                    'Kaynak': 'TCMB Doğrudan Veri'
                })

    # DataFrame'e dönüştür ve ekrana yazdır
    df_tum_caprazlar = pd.DataFrame(capraz_kur_listesi)
    display(df_tum_caprazlar)


except Exception as e:
    print("Çapraz kurlar listelenirken bir hata oluştu:", e)

,Para Birimi,Tanım,Çapraz Kur Tipi,Değer,Kaynak
0,AUD,AVUSTRALYA DOLARI,AUD/USD,1.46470,TCMB Doğrudan Veri
1,DKK,DANİMARKA KRONU,DKK/USD,6.74100,TCMB Doğrudan Veri
2,EUR,EURO,USD/EUR,1.10900,TCMB Doğrudan Veri
3,GBP,İNGİLİZ STERLİNİ,USD/GBP,1.32690,TCMB Doğrudan Veri
4,CHF,İSVİÇRE FRANGI,CHF/USD,0.83210,TCMB Doğrudan Veri
5,SEK,İSVEÇ KRONU,SEK/USD,9.96690,TCMB Doğrudan Veri
6,CAD,KANADA DOLARI,CAD/USD,1.42530,TCMB Doğrudan Veri
7,KWD,KUVEYT DİNARI,USD/KWD,3.24040,TCMB Doğrudan Veri
8,NOK,NORVEÇ KRONU,NOK/USD,9.55470,TCMB Doğrudan Veri
9,SAR,SUUDİ ARABİSTAN RİYALİ,SAR/USD,3.75300,TCMB Doğrudan Veri


In [46]:
display(df_tum_caprazlar)

,Para Birimi,Tanım,Çapraz Kur Tipi,Değer,Kaynak
0,AUD,AVUSTRALYA DOLARI,AUD/USD,1.464700,TCMB Doğrudan Veri
1,DKK,DANİMARKA KRONU,DKK/USD,6.741000,TCMB Doğrudan Veri
2,EUR,EURO,USD/EUR,0.901713,TCMB Doğrudan Veri
3,GBP,İNGİLİZ STERLİNİ,USD/GBP,1.326900,TCMB Doğrudan Veri
4,CHF,İSVİÇRE FRANGI,CHF/USD,0.832100,TCMB Doğrudan Veri
5,SEK,İSVEÇ KRONU,SEK/USD,9.966900,TCMB Doğrudan Veri
6,CAD,KANADA DOLARI,CAD/USD,1.425300,TCMB Doğrudan Veri
7,KWD,KUVEYT DİNARI,USD/KWD,3.240400,TCMB Doğrudan Veri
8,NOK,NORVEÇ KRONU,NOK/USD,9.554700,TCMB Doğrudan Veri
9,SAR,SUUDİ ARABİSTAN RİYALİ,SAR/USD,3.753000,TCMB Doğrudan Veri


In [ ]:
import pandas as pd

try:
    # TCMB günlük kur XML adresi
    url = "https://www.tcmb.gov.tr/kurlar/today.xml"
    df = pd.read_xml(url, xpath="//Currency")

    # İhtiyacımız olan dövizlerin kurlarını sözlük olarak alalım (TRY karşılıkları)
    # TCMB kurları genellikle 1, 100 veya 1000 birim üzerinden verilir (Unit sütununa dikkat ederek normalize ediyoruz)
    kurlar = {}
    for _, row in df.iterrows():
        code = row.get('CurrencyCode')
        buying = row.get('ForexBuying')
        unit = row.get('Unit', 1)
        if code and buying:
            # 1 birim dövizin TL karşılığı
            kurlar[code] = float(buying) / float(unit)

    # TL zaten 1'dir
    kurlar['TRY'] = 1.0

    # İstenen kurların hesaplanması
    # USD/TRY
    usd_try = kurlar.get('USD', 0)



    # Sonuçları DataFrame olarak düzenleyelim
    sonuc_data = {
        'Parite': ['USD/TRY', 'USD/EUR', 'USD/RUB', 'USD/CHF', 'USD/GBP'],
        'Değer': [usd_try, eur_usd, rub_usd, chf_usd, gbp_usd]
    }

    df_sonuc = pd.DataFrame(sonuc_data)
    display(df_sonuc)

except Exception as e:
    print("Hesaplama yapılırken bir hata oluştu:", e)

,Parite,Değer
0,USD/TRY,50.169100
1,USD/EUR,0.901753
2,USD/RUB,83.201930
3,USD/CHF,0.834028
4,USD/GBP,0.754925


In [ ]:
me2n.head(10)

,ГОД,Документ закупки,Позиция,Создал,Группа закупок,Закуп. организация,Индикатор удаления,Дата документа,Наш знак,Ваш код,...,Тип неполноты данных,Группа материалов,Материал_ME2N,Материал_ME5A,Условие платежа,Номер заявки на платеж,Оплачиваемая сумма,Валюта_zpp001,Оплаченная Сумма_zpp001,сумма Оплаченная Сумма_zpp001
0,2026,3100016873,10,KGAIVUK,2,2992,NaN,2026-12-03,AK-1734,AK-1734,...,Документ полностью сохранен,MZ-4999,4500527883,4.500528e+09,0001,NaN,NaN,NaN,NaN,0.0
1,2026,3100017736,10,MMASLOV,3,2993,NaN,2026-11-15,Спецификация 908,АК-Р-1825,...,Документ полностью сохранен,MZ-4999,4500187031,4.500187e+09,Z000,NaN,NaN,NaN,NaN,NaN
2,2026,3100017736,30,MMASLOV,3,2993,NaN,2026-11-15,Спецификация 908,АК-Р-1825,...,Документ полностью сохранен,MZ-4999,4500198122,4.500198e+09,Z000,NaN,NaN,NaN,NaN,NaN
3,2026,3100017736,20,MMASLOV,3,2993,NaN,2026-11-15,Спецификация 908,АК-Р-1825,...,Документ полностью сохранен,MZ-4999,4500198114,4.500198e+09,Z000,NaN,NaN,NaN,NaN,NaN
4,2026,3100017736,50,MMASLOV,3,2993,NaN,2026-11-15,Спецификация 908,АК-Р-1825,...,Документ полностью сохранен,MZ-4999,4500198669,4.500199e+09,Z000,NaN,NaN,NaN,NaN,NaN
5,2026,3100017736,60,MMASLOV,3,2993,NaN,2026-11-15,Спецификация 908,АК-Р-1825,...,Документ полностью сохранен,MZ-4999,4500193730,4.500194e+09,Z000,NaN,NaN,NaN,NaN,NaN
6,2026,3100017736,40,MMASLOV,3,2993,NaN,2026-11-15,Спецификация 908,АК-Р-1825,...,Документ полностью сохранен,MZ-4999,4500198126,4.500198e+09,Z000,NaN,NaN,NaN,NaN,NaN
7,2026,3100018411,30,NOZKAN,3,2993,NaN,2026-11-06,AKP2096,AKP2096,...,Документ полностью сохранен,MZ-6003,4500136085,4.500086e+09,Z000,NaN,NaN,NaN,NaN,NaN
8,2026,3100018411,20,NOZKAN,3,2993,NaN,2026-11-06,AKP2096,AKP2096,...,Документ полностью сохранен,MZ-6008,4500566068,4.500368e+09,Z000,NaN,NaN,NaN,NaN,NaN
9,2026,3100018409,40,IMAMI,3,2993,NaN,2026-11-06,SP 783,AK-P-925.1,...,Документ полностью сохранен,MZ-4015,4500186989,4.500187e+09,Z000,NaN,NaN,NaN,NaN,NaN
